In [ ]:
from atlas.common.config.loader import get_settings
from atlas.common.paths.loader import get_paths
from atlas.common.spark.session import get_spark_session

settings = get_settings("configs/base.yaml", "configs/local.yaml", "pyproject.toml")

In [ ]:
spark = get_spark_session(settings.spark, settings.storage, settings.application.name)

In [ ]:
paths = get_paths(settings)

In [ ]:
bronze_customer_path = paths.bronze_path("customer/cdc/customers/job")

In [ ]:
customer_bronze_data = spark.read.format("parquet").load(bronze_customer_path)

In [ ]:
from pyspark.sql import functions as F

customer_bronze_data.select(F.col("raw_key"), F.col("raw_value")).show(vertical=True, truncate=False, n=1)

In [ ]:
from pyspark.sql.types import LongType, StringType, StructField, StructType

customer_record_schema = StructType([
    StructField("customer_id", LongType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("email", StringType(), True),
    StructField("phone_number", StringType(), True),
    StructField("date_of_birth", LongType(), True),
    StructField("status", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("created_at", StringType(), False),
    StructField("updated_at", StringType(), False),
])

In [ ]:
customer_source_schema = StructType([
    StructField("version", StringType(), False),
    StructField("connector", StringType(), False),
    StructField("name", StringType(), False),
    StructField("ts_ms", LongType(), False),
    StructField("snapshot", StringType(), True),
    StructField("db", StringType(), False),
    StructField("sequence", StringType(), True),
    StructField("ts_us", LongType(), True),
    StructField("ts_ns", LongType(), True),
    StructField("schema", StringType(), False),
    StructField("table", StringType(), False),
    StructField("txId", LongType(), True),
    StructField("lsn", LongType(), True),
    StructField("xmin", LongType(), True),
])

In [ ]:
customer_payload_schema = StructType([
    StructField("before", customer_record_schema, True),
    StructField("after", customer_record_schema, True),
    StructField("source", customer_source_schema, False),
    StructField("op", StringType(), False),
    StructField("ts_ms", LongType(), False),
    StructField("ts_us", LongType(), True),
    StructField("ts_ns", LongType(), True),
])

In [ ]:
customer_debezium_schema = StructType([
    StructField("payload", customer_payload_schema, True),
])

In [ ]:
customer_parsed_data = customer_bronze_data.withColumn("debezium", F.from_json(F.col("raw_value")
                                                                               , customer_debezium_schema))

In [ ]:
customer_parsed_data.select(
    "debezium.payload.before",
    "debezium.payload.after",
    "debezium.payload.op",
    "debezium.payload.source.lsn",
    "kafka_partition",
    "kafka_offset",
).show(1, truncate=False, vertical=True)

In [ ]:
customer_cdc_data = customer_parsed_data.withColumn("customer",
                                                    F.when(
                                                        F.col("debezium.payload.op") == "d",
                                                        F.col("debezium.payload.before"),

                                                    ).otherwise(
                                                        F.col("debezium.payload.after")
                                                    )
                                                    )

In [ ]:
customer_cdc_data_normalized = customer_cdc_data.select(
    F.col("customer.customer_id").alias("customer_id"),
    F.col("customer.first_name").alias("first_name"),
    F.col("customer.last_name").alias("last_name"),
    F.col("customer.email").alias("email"),
    F.col("customer.phone_number").alias("phone_number"),
    F.date_add(
        F.lit("1970-01-01").cast("date"),
        F.col("customer.date_of_birth").cast("int")
    ).alias("date_of_birth"),
    F.col("customer.status").alias("status"),
    F.col("customer.segment").alias("segment"),
    F.try_to_timestamp(F.col("customer.created_at")).alias("created_at"),
    F.try_to_timestamp(F.col("customer.updated_at")).alias("updated_at"),
    F.timestamp_millis(F.col("debezium.payload.ts_ms")).alias("cdc_timestamp"),
    F.timestamp_millis(F.col("debezium.payload.source.ts_ms")).alias("source_timestamp"),
    F.col("debezium.payload.op").alias("cdc_operation"),
    F.col("debezium.payload.source.lsn").alias("source_lsn"),
    F.col("kafka_partition").alias("kafka_partition"),
    F.col("kafka_offset").alias("kafka_offset"),
    F.col("kafka_timestamp").alias("kafka_timestamp"),
    F.col("is_tombstone").alias("is_tombstone"),
    F.col("ingested_at").alias("ingested_at"),
)

In [ ]:
customer_cdc_data_normalized.show(vertical=True, truncate=False, n=1)